# Phase 4 — Final Run (r=16, full data, 3 epochs)

Projected ~10 h vs the 12 h background ceiling — slim margin, so this notebook
has a **resume lifeboat**: if the run dies at the ceiling, attach the dead
version's Output as an input dataset, set `RESUME_INPUT` in Cell 1, and
commit again. train.py picks up from the last checkpoint automatically.

Run Cells 1–4 interactively to verify setup, then Save Version →
**Save & Run All (Commit)**.

In [ ]:
# CELL 1 — config
import os
os.environ["REPO_URL"]        = "https://github.com/Rick-Roy-JC/text-to-sql-qlora-v2.git"
os.environ["DATA_DATASET"]    = "/kaggle/input/datasets/aritraroy3/spider-prepared-v2"
os.environ["CHECKPOINT_ROOT"] = "/kaggle/working/checkpoints"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # single run -> single GPU

# LIFEBOAT: normally empty. If resuming a dead run, attach that version's
# Output as an input dataset and put its path here, e.g.
# "/kaggle/input/phase4-final-run"  (check the real mount with: !ls /kaggle/input/)
os.environ["RESUME_INPUT"] = ""

In [ ]:
# CELL 2 — hardware tripwire
!nvidia-smi --query-gpu=index,name,compute_cap --format=csv

In [ ]:
# CELL 3 — pinned installs (VERSIONS.txt)
%pip install -q unsloth==2026.6.9 trl==0.24.0 peft==0.19.1 transformers==5.5.0
import unsloth, trl, peft, transformers, torch
print("unsloth", unsloth.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| transformers", transformers.__version__)

In [ ]:
# CELL 4 — repo + data + lifeboat restore (all idempotent)
import os, shutil
%cd /kaggle/working
if not os.path.exists("text-to-sql-qlora-v2"):
    !git clone $REPO_URL
%cd text-to-sql-qlora-v2
!git pull
!mkdir -p data && cp $DATA_DATASET/*.jsonl data/
!ls -la data/

resume_input = os.environ.get("RESUME_INPUT", "")
if resume_input:
    src = os.path.join(resume_input, "checkpoints", "r16")
    dst = os.path.join(os.environ["CHECKPOINT_ROOT"], "r16")
    print(f"LIFEBOAT: restoring checkpoints {src} -> {dst}")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    !ls $CHECKPOINT_ROOT/r16/
else:
    print("fresh run (no RESUME_INPUT set)")

In [ ]:
# CELL 5 — THE FINAL RUN. Full train.jsonl (~6.3k), 3 epochs, r=16.
# eval_steps=250 / save_steps=100: frequent-enough saves for the lifeboat,
# infrequent-enough evals that they don't eat hours (47 evals would have).
!python src/train.py --rank 16 --epochs 3 \
    --eval_steps 250 --save_steps 100 \
    --output_root $CHECKPOINT_ROOT

In [ ]:
# CELL 6 — harvest: eval curve, adapter zip
import json, glob, re, os
ROOT = os.environ["CHECKPOINT_ROOT"]
states = sorted(glob.glob(f"{ROOT}/r16/checkpoint-*/trainer_state.json"),
                key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)))
hist = json.load(open(states[-1]))["log_history"]
print("step   eval_loss")
for h in hist:
    if "eval_loss" in h:
        print(f"{h['step']:<7}{h['eval_loss']:.4f}")

!cd $CHECKPOINT_ROOT && zip -rq /kaggle/working/final_adapter_r16.zip r16/adapter_final
!ls -la /kaggle/working/final_adapter_r16.zip
print("Download from Output tab -> commit to git as adapters/final_r16/")